# Week 3, Notebook 3: Deploy Your Model as a Web App
## From Notebook to Full-Stack — The Finish Line

**What you'll build:** A Gradio web app where anyone can interact with your VAE.

**Time estimate:** 30 minutes

---
### The Deployment Stack
```
Your VAE Model (PyTorch)
        ↓
   Gradio Interface (Python)
        ↓
   Web App (auto-generated)
        ↓
   Hugging Face Spaces (free hosting)
```

## Part 1: Save Your Trained Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ============================================================
# Recreate and train the VAE (self-contained for deployment)
# ============================================================
class VAE(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid(),
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# Quick train for this notebook (copy from W3_02 in practice)
print("Training a quick VAE for deployment demo...")

def generate_digit_dataset_quick(n_per_digit=200):
    images, labels = [], []
    for digit in range(10):
        for _ in range(n_per_digit):
            img = np.random.rand(8, 8) * 0.1
            if digit == 0:
                img[1:7, 1] = 0.8; img[1:7, 6] = 0.8; img[1, 1:7] = 0.8; img[6, 1:7] = 0.8
            elif digit == 1:
                img[1:7, 4] = 0.9
            elif digit == 7:
                img[1, 1:7] = 0.8; img[1:7, 6] = 0.8
            else:
                # Generic pattern for other digits
                np.random.seed(digit * 100 + _)
                img[1:7, 1:7] = np.random.rand(6, 6) * 0.3 + 0.3
            img += np.random.randn(8, 8) * 0.05
            img = np.clip(img, 0, 1)
            images.append(img)
            labels.append(digit)
    return np.array(images, dtype=np.float32), np.array(labels)

X_imgs, y_labels = generate_digit_dataset_quick()
X_flat = torch.FloatTensor(X_imgs.reshape(-1, 64))

vae = VAE()
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(80):
    idx = torch.randperm(len(X_flat))[:64]
    x_hat, mu, logvar = vae(X_flat[idx])
    recon = F.mse_loss(x_hat, X_flat[idx], reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    loss = recon + 0.5 * kl
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Save the model
torch.save(vae.state_dict(), 'vae_model.pth')
print("✓ Model saved to vae_model.pth")

## Part 2: Create the Gradio App

Gradio turns any Python function into a web interface.  
Install: `pip install gradio`

In [ ]:
# ============================================================
# Gradio app code — save this as app.py for deployment
# ============================================================
# The app.py file is generated below using write().
# It contains a full Gradio interface for the VAE.

app_lines = [
    "import torch",
    "import torch.nn as nn",
    "import numpy as np",
    "import matplotlib",
    'matplotlib.use("Agg")',
    "import matplotlib.pyplot as plt",
    "import gradio as gr",
    "",
    "# --- VAE Model Definition ---",
    "class VAE(nn.Module):",
    "    def __init__(self, input_dim=64, hidden_dim=128, latent_dim=2):",
    "        super().__init__()",
    "        self.encoder = nn.Sequential(",
    "            nn.Linear(input_dim, hidden_dim), nn.ReLU(),",
    "            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU())",
    "        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)",
    "        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)",
    "        self.decoder = nn.Sequential(",
    "            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),",
    "            nn.Linear(hidden_dim // 2, hidden_dim), nn.ReLU(),",
    "            nn.Linear(hidden_dim, input_dim), nn.Sigmoid())",
    "    def decode(self, z): return self.decoder(z)",
    "",
    "model = VAE()",
    'model.load_state_dict(torch.load("vae_model.pth", map_location="cpu"))',
    "model.eval()",
    "",
    "def generate_digits(z1, z2, n_samples):",
    "    fig, axes = plt.subplots(1, max(1,int(n_samples)), figsize=(2*n_samples,2))",
    "    if n_samples == 1: axes = [axes]",
    "    for i, ax in enumerate(axes):",
    "        z = torch.FloatTensor([[z1, z2]]) if i == 0 else torch.randn(1, 2)",
    "        with torch.no_grad(): gen = model.decode(z).numpy().reshape(8, 8)",
    "        ax.imshow(gen, cmap='gray_r', vmin=0, vmax=1); ax.axis('off')",
    "    plt.tight_layout()",
    "    return fig",
    "",
    "demo = gr.Interface(fn=generate_digits,",
    "    inputs=[gr.Slider(-3,3,value=0,label='z1'),",
    "            gr.Slider(-3,3,value=0,label='z2'),",
    "            gr.Slider(1,8,value=4,step=1,label='Samples')],",
    "    outputs=gr.Plot(), title='VAE Digit Generator')",
    "demo.launch()",
]

with open("app.py", "w") as f:
    f.write("\n".join(app_lines))

print("\u2713 app.py saved!")
print("\nTo run locally:")
print("  1. pip install gradio")
print("  2. python app.py")
print("\nTo deploy on Hugging Face Spaces:")
print("  1. Create a new Space at huggingface.co/new-space")
print("  2. Upload: app.py, vae_model.pth, requirements.txt")
print("  3. Your app is live!")

## Part 3: Prepare for Deployment

Create the files needed for Hugging Face Spaces deployment.

In [ ]:
# ============================================================
# Create requirements.txt and README
# ============================================================
req_lines = ["torch", "gradio", "numpy", "matplotlib"]
with open("requirements.txt", "w") as f:
    f.write("\n".join(req_lines))

readme_lines = [
    "# VAE Digit Generator",
    "",
    "A Variational Autoencoder trained to generate handwritten digit patterns.",
    "",
    "## What This Does",
    "- Generate: Create new digit-like images by exploring the 2D latent space",
    "- Interpolate: See smooth transitions between different generated images",
    "",
    "## Built With",
    "- PyTorch (model training)",
    "- Gradio (web interface)",
    "- Hugging Face Spaces (hosting)",
    "",
    "## Run Locally",
    "pip install -r requirements.txt",
    "python app.py",
]
with open("README.md", "w") as f:
    f.write("\n".join(readme_lines))

print("Deployment files created:")
print("  - app.py          (Gradio web app)")
print("  - vae_model.pth   (trained model weights)")
print("  - requirements.txt (dependencies)")
print("  - README.md        (project description)")
print()
print("=" * 60)
print("DEPLOYMENT CHECKLIST:")
print("=" * 60)
print("1. Go to https://huggingface.co/new-space")
print("2. Choose Gradio as the SDK")
print("3. Upload all 4 files above")
print("4. Wait ~2 minutes for it to build")
print("5. Share the URL -- your app is LIVE!")
print()
print("You now have a DEPLOYED generative AI application.")
print("This is your portfolio piece.")

## 🏆 Congratulations — You Completed the 3-Week Plan!

### What You Built:
1. **Week 1:** Neural network from scratch (pure Python) — forward pass, backprop, ReLU
2. **Week 2:** Deep networks in PyTorch — initialization, BatchNorm, loss landscapes
3. **Week 3:** Generative AI — VAE from scratch AND in PyTorch, deployed as a web app

### Curriculum Points Mastered:
- ✅ ① Universal Approximation Theorem (width experiments)
- ✅ ② ReLU changed everything (sigmoid vs ReLU comparison)
- ✅ ③ Overparameterization helps (parameter scaling experiment)
- ✅ ④ Neural networks minimize loss, not "understand" (built it yourself!)
- ✅ ⑤ Backpropagation is just the chain rule (gradient checking proved it)
- ✅ ⑥ Geometry matters (loss landscape visualization)
- ✅ ⑦ Capacity ≠ performance (depth + init + norm experiments)

### Your Portfolio Now Has:
- [ ] GitHub repo with all notebooks
- [ ] Deployed Gradio app on Hugging Face
- [ ] Visualizations you can explain to anyone
- [ ] Code you wrote, not just imported

### What's Next?
1. **CNNs** → Apply to real image datasets (CIFAR-10, ImageNet)
2. **Transformers** → The architecture behind GPT, BERT, and modern LLMs
3. **Diffusion Models** → State-of-the-art image generation
4. **Fine-tuning LLMs** → Hugging Face makes this accessible

*"You don't truly understand something until you can build it from scratch."*  
*You just did.*

## Visualizing the Deployment Architecture

A diagram showing how the user interacts with the Gradio web UI to trigger inference on the VAE Decoder.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
pos = {
    "User (Web UI)": (0, 0),
    "Gradio App": (2, 0),
    "Latent Vector (z)": (4, 0),
    "VAE Decoder\n(Inference)": (6, 0),
    "Output Image": (8, 0)
}

G.add_edges_from([
    ("User (Web UI)", "Gradio App"),
    ("Gradio App", "Latent Vector (z)"),
    ("Latent Vector (z)", "VAE Decoder\n(Inference)"),
    ("VAE Decoder\n(Inference)", "Output Image"),
    ("Output Image", "User (Web UI)") # Feedback loop
])

plt.figure(figsize=(12, 3))
nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=4000, node_shape="o", font_size=9)
plt.title("Deployed VAE Inference Flow")

plt.savefig('w3_03_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
